# Run all experiments on Colab (GPU)
**Runtime → Change runtime type → T4 GPU**, then *Run all*.

Results are written to Google Drive (`MyDrive/CS545_GAT/results`) so nothing is lost if Colab disconnects —
just re-run; finished runs are skipped. At the end, the results are zipped for download and can be committed to
the repo's `results/` folder, which the report notebook (`GAT_Project.ipynb`) reads.

**If Colab disconnects or restarts:** re-run the setup cells (GPU check, clone, Drive mount) before any experiment cell, so the project folder and `OUT` are defined again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = "https://github.com/YashDolorase/cs545-gat.git"
import os
if not os.path.exists("/content/project"):
    !git clone -q $REPO_URL /content/project
%cd /content/project
!git pull -q
!pip -q install torch_geometric
import torch, torch_geometric
print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| CUDA", torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/CS545_GAT/results"
os.makedirs(OUT, exist_ok=True)
print(OUT)

## Sanity checks (from-scratch GAT layer == PyG `GATConv`, noise injection, attention stats)

In [ ]:
import os; os.chdir("/content/project")  # Colab resets the working dir on reconnect
!python -m pytest tests -q -W ignore

## E1 — Benchmark: MLP, GCN, GraphSAGE, GAT on Cora + CiteSeer, 10 seeds

In [ ]:
import os; os.chdir("/content/project")  # Colab resets the working dir on reconnect
!python -W ignore scripts/run_experiments.py --exp e1 --seeds 10 --out $OUT

## E3 — Noisy neighbours: 0/10/25/50/100% random edges on Cora, 10 seeds

In [ ]:
import os; os.chdir("/content/project")  # Colab resets the working dir on reconnect
!python -W ignore scripts/run_experiments.py --exp e3 --seeds 10 --out $OUT

## Stretch — GATv2, and low-homophily graphs (Texas, Chameleon)
Run only after E1 and E3 are done.

In [ ]:
import os; os.chdir("/content/project")  # Colab resets the working dir on reconnect
RUN_STRETCH = False
if RUN_STRETCH:
    !python -W ignore scripts/run_experiments.py --exp e1 --models GATv2 --datasets Cora CiteSeer --out $OUT
    !python -W ignore scripts/run_experiments.py --exp e3 --models GATv2 --out $OUT
    !python -W ignore scripts/run_experiments.py --exp e1 --models MLP GCN SAGE GAT GATv2 --datasets Texas Chameleon --out $OUT

## Quick look at the results

In [ ]:
import pandas as pd
e1 = pd.read_csv(f"{OUT}/e1_benchmark.csv")
display((e1.groupby(["dataset", "model"])["test_acc"].agg(["mean", "std", "count"]) * [100, 100, 1]).round(2))
if os.path.exists(f"{OUT}/e3_noise.csv"):
    e3 = pd.read_csv(f"{OUT}/e3_noise.csv")
    display((e3.pivot_table(index="noise", columns="model", values="test_acc", aggfunc="mean") * 100).round(1))

In [ ]:
!cd $OUT/.. && zip -qr /content/results.zip results
from google.colab import files
files.download("/content/results.zip")